In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import os

from src.preprocessing import preprocess_stable, train_val_split
from src.models import StableLogisticRegression, StableLogisticRegressionVis, DecisionTreeScratch, f1_macro
from src.visualization import plot_logistic_regression_training, print_decision_tree

np.random.seed(42)

path = kagglehub.competition_download('ai-lab-recruitment-task-2')

train_df = pd.read_csv(os.path.join(path, 'train.csv'))
test_df = pd.read_csv(os.path.join(path, 'test.csv'))

X_raw = train_df.drop(columns=['person_id', 'loan_status'])
y = train_df['loan_status'].values
test_ids = test_df['person_id'].values
X_test_raw = test_df.drop(columns=['person_id'])

In [ ]:
X, scalers = preprocess_stable(X_raw, is_train=True)
X_test, _ = preprocess_stable(X_test_raw, is_train=False, scalers=scalers)

X_train, X_val, y_train, y_val = train_val_split(X, y, val_ratio=0.15)

In [ ]:
model_lr = StableLogisticRegression(learning_rate=0.8, iterations=25000, lambda_reg=0.00005)
model_lr.fit(X_train, y_train)
val_probs = model_lr.predict_proba(X_val)

best_threshold = 0.5
best_f1 = 0

for t in np.arange(0.1, 0.9, 0.01):
    y_val_pred_t = (val_probs >= t).astype(int)
    current_f1 = f1_macro(y_val, y_val_pred_t)

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = t

print(f"Optimal Threshold ditemukan: {best_threshold:.2f}")
print(f"Macro F1-Score pada Validation Set: {best_f1:.4f}")

In [ ]:
final_model = StableLogisticRegression(learning_rate=0.8, iterations=30000, lambda_reg=0.00005)
final_model.fit(X, y)

test_probs = final_model.predict_proba(X_test)
test_predictions = (test_probs >= best_threshold).astype(int)

submission_df = pd.DataFrame({
    'person_id': test_ids,
    'loan_status': test_predictions
})

submission_df.to_csv('submission_final_stable.csv', index=False)

In [ ]:
model_lr_vis = StableLogisticRegressionVis(learning_rate=0.8, iterations=10000, lambda_reg=0.00005)
model_lr_vis.fit(X_train, y_train)

plot_logistic_regression_training(model_lr_vis)

In [ ]:
print("Melatih Decision Tree...")
dt_model = DecisionTreeScratch(max_depth=3)
dt_model.fit(X_train, y_train)

print("\n=== GAMBAR PERCABANGAN DECISION TREE ===")
print_decision_tree(dt_model.root)